# Instruction fine-tuning
**Vs pretraining:** train on prompts + answers; same next-token loss (ignore padding); evaluate answer quality.

| Goal | Run |
|---|---|
| Train from pretrained weights | Under **Run folder**, set `checkpoint_path = None` and comment out the latest-checkpoint line. Then **Run All**. |
| Resume training | Under **Run folder**, uncomment the latest-checkpoint line, then **Run All**. `num_epochs` adds more epochs. |
| Generate from a checkpoint | Under **Run folder**, uncomment the latest-checkpoint line. Run **Goal & config → Load model → Data + DataLoader**, then skip to **Extract model responses & save → Evaluate & reflect**. |
| Grade saved responses only | Use the Mac kernel. Run **Goal & config → `format_input` function cell only → Evaluate & reflect**. |

Restart the kernel when starting a different experiment.

## 0 · Goal & config
- **Mac:** local `.venv` kernel; run from `Training/`.
- **Runpod:** keep [setup.ipynb](../Runpod/setup.ipynb) on the local kernel and run **Start**. Here, choose **Select Kernel → Select Another Kernel → Existing Jupyter Server**, paste the URL, then choose **Python 3**. The first code cell sets the folder; device should print `cuda`.
- **Resume on Runpod:** upload the checkpoint with setup's `upload_checkpoint(...)` before selecting it below.

In [1]:
from pathlib import Path

# Use the copied project folder on Runpod.
pod_folder = Path("/workspace/LLM-fine-tuning/Training")
if pod_folder.is_dir():
    %cd {pod_folder}

#### Imports

In [2]:
import json
from datetime import datetime, timezone
from functools import partial

import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs
from reference.data import load_instruction_data
from reference.evals import query_model
from reference.generate import generate
from reference.losses import calc_loss_loader
from reference.model_io import download_weights, text_to_token_ids, token_ids_to_text
from reference.models.gpt2 import GPT2Model, load_weights_into_gpt
from reference.plot import plot_losses
from reference.train import train_model_simple

#### Model config

In [3]:
# Options: gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M).
CHOOSE_MODEL = "gpt2-medium (355M)"

# ** unpacks dictionary entries; later values win on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}

# Hugging Face calls the small model "gpt2".
model_repo = "openai-community/" + CHOOSE_MODEL.split()[0].replace("gpt2-small", "gpt2")

#### Run folder
One folder per experiment; checkpoint and response filenames share a UTC timestamp (same clock on Mac and Runpod).

In [ ]:
folder = Path("runs/instruction/002_finetuned")
checkpoint_path = None  # Start from pretrained weights.
# checkpoint_path = max(folder.glob("checkpoint_*.pth"))  # Resume the latest checkpoint.

# For grading only: latest responses. Loading/saving a checkpoint selects its matching JSON.
responses_file = max(folder.glob("responses_*.json"), default=None)

#### Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("device:", device)

#### Tokenizer

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

## 1 · Load model
Load pretrained weights, or restore a checkpoint's model and optimizer.

In [ ]:
%%capture

if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    # Older checkpoints use the model config set above.
    model_config = checkpoint.get("model_config", model_config)

model = GPT2Model(model_config)
if checkpoint_path is None:
    load_weights_into_gpt(model, download_weights(model_repo))
else:
    model.load_state_dict(checkpoint["model"])
    responses_file = checkpoint_path.with_name(
        checkpoint_path.stem.replace("checkpoint_", "responses_") + ".json"
    )

model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
if checkpoint_path is not None:
    optimizer.load_state_dict(checkpoint["optimizer"])
model.eval()

## 2 · Data + DataLoader
Load data, prepare token IDs, and build batches.

| Input tokens | Training target |
|---|---|
| `[P1, P2, R1, R2]` | `[P2, R1, R2, EOS]` |

`P` = prompt token; `R` = response token; `EOS` = end-of-text.
`collate_fn` builds these shifted targets; padding is ignored in the loss.

In [ ]:
data = load_instruction_data()
print("Instruction Data Keys: ", data[0].keys())
data[:1]

### Split data: 85% train, 5% validation, 10% test

In [ ]:
train_portion = int(0.85 * len(data))
test_portion  = int(0.10 * len(data))
val_portion   = int(len(data) - train_portion - test_portion)

train_data = data[:train_portion]
test_data  = data[train_portion:train_portion+test_portion]
val_data   = data[train_portion+test_portion:]


print("Fields:", train_data[0].keys())
print(f"train Length: {len(train_data)}")
print(f"test Length: {len(test_data)}")
print(f"val Length: {len(val_data)}")

### Prompt template
Combines an example's instruction and optional input into one text prompt. The response is added separately.

In [ ]:
def format_input(entry):

  instruction_text = (
      f"Below is an instruction that describes a task. "
      f"Write a response that appropriately completes the request."
      f"\n\n### Instruction:\n{entry['instruction']}"
  )

  input_text = (
  f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
  )

  return instruction_text + input_text

In [ ]:
model_input = format_input(data[50])
detailed_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + detailed_response)

### Dataset
Dataset formats and tokenizes each example; `collate_fn` creates inputs and next-token targets.

In [ ]:
class InstructionDataset(Dataset):

    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            formatted_input    = format_input(entry)
            formatted_response = f"\n\n### Response:\n{entry['output']}"
            combined_text = formatted_input + formatted_response

            # List of token IDs, e.g. [10, 20, 30]; collate adds padding.
            encoded_text = tokenizer.encode(combined_text)
            self.encoded_texts.append(encoded_text)


    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)


# Example
dataset = InstructionDataset(train_data, tokenizer)
print(f"Training Dataset Length: {len(dataset)} -> same as train_data")
print(f"dataset[0] ({len(dataset[0])} tokens): {dataset[0]}")
print(f"dataset[1] ({len(dataset[1])} tokens): {dataset[1]}")

### Custom collate function
In pretraining, `calc_loss_batch` moves batches to the device. Here, `collate_fn` does this before returning each batch. With `num_workers=0`, it runs in the main process.

In [ ]:
def custom_collate_fn(batch, pad_token_id, ignore_index = -100, device = "cpu", allowed_max_length = None):

    batch_max_length = max(len(item)+ 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs  = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # mask = torch.tensor([False, False, True, True])
        mask    = targets == pad_token_id

        # torch.nonzero(mask) tensor([[2], [3]]) --> squeeze() tensor([2, 3])
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            # targets[torch.tensor([3])] = ignore_index
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Stack the batch and move it to the device.
    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


# Example - 2 examples in batch
batch = [dataset[0], dataset[1]] # expects list of examples
input_batch, target_batch = custom_collate_fn(batch, pad_token_id=tokenizer.eot_token)
print(f"Same padded length for every example in this batch:")
print(f"Each input:  {input_batch.shape[1]} tokens")
print(f"Each target: {target_batch.shape[1]} tokens\n")

# Example - Inputs versus targets
print(f"inputs[0]:  {input_batch[0][:5].tolist()} ... {input_batch[0][-5:].tolist()}")
print(f"targets[0]: {target_batch[0][:5].tolist()} ...  {target_batch[0][-5:].tolist()}\n")

print(f"Input tokens in entire batch:  {input_batch.numel()}")
print(f"Target tokens in entire batch: {target_batch.numel()}")

> **Loss mask:** keep the first end-of-text target; ignore extra padding targets (`-100`). Prompt and response tokens both contribute to the loss. Response-only training would also mask prompt targets.

In [ ]:
pad_token_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})

customized_collate_fn = partial(custom_collate_fn,
                               device=device,
                               allowed_max_length=model_config["context_length"],
                               pad_token_id=pad_token_id[0]) 

# Use 50256, not [50256]; collate adds the brackets.

### DataLoader
Padding gives all examples the same length within a batch. Length can vary between batches.

In [ ]:
num_workers = 0
batch_size  = 8
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
    )

# Inspect one batch; its padded length can change when shuffled.
inputs, targets = next(iter(train_loader))
print(f"inputs.shape:  {inputs.shape}\ntargets.shape: {targets.shape}\n")

print(f"inputs[0]:  {inputs[0][:5].tolist()} ... {inputs[0][-5:].tolist()}")
print(f"targets[0]: {targets[0][:5].tolist()} ...  {targets[0][-5:].tolist()}")

In [ ]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3 · Loss / objective
Next-token cross-entropy: padding targets are ignored; prompt and response tokens contribute.

In [ ]:
# Loss before this training run

torch.manual_seed(123)
model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss before training:",   train_loss)
print("Validation loss before training:", val_loss)

## 4 · Generate / predict
Generate a starting response before this training run.

In [ ]:
torch.manual_seed(123)
model.eval()
sample_input = format_input(val_data[0])
token_ids = generate(model, 
idx = text_to_token_ids(sample_input, tokenizer).to(device), 
max_new_tokens = 35,
context_length = model_config["context_length"],
eos_id = tokenizer.eot_token)

generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(sample_input):].strip()
print(f"{response_text}")

## 5 · Train
Run this cell again to continue with the current model and optimizer. Epoch/step counters restart each call.

In [ ]:
%%time

torch.manual_seed(123)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, optimizer, device, tokenizer,
    train_loader, val_loader,
    eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), num_epoch=num_epochs
)

### Plot Losses

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

## 6 · Save checkpoint & responses
**Runpod → Mac:** open setup's Jupyter URL in your browser and download this run's responses JSON for grading; download the checkpoint too if you want to resume. Keep the same run folder on your Mac.

Download before running **Delete → Confirm** in setup (local kernel). Deleting the pod also deletes its files.

#### Save model & optimizer

In [ ]:
folder.mkdir(parents=True, exist_ok=True)

latest = folder / f"checkpoint_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}.pth"
checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "model_config": model_config,
}
torch.save(checkpoint, latest)
responses_file = latest.with_name(latest.stem.replace("checkpoint_", "responses_") + ".json")

#### Extract model responses & save

In [ ]:
def extract_response_and_save(path, file_name):
    model.eval()
    for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
        input_text = format_input(entry)
        token_ids = generate(
            model,
            idx=text_to_token_ids(input_text, tokenizer).to(device),
            max_new_tokens=256,
            context_length=model_config["context_length"],
            eos_id=tokenizer.eot_token
        )

        generated_text = token_ids_to_text(token_ids, tokenizer)
        response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
        test_data[i]["model_response"] = response_text
        entry.pop("judge_score", None)  # A new response needs a new score.

    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    with open(path / file_name, "w") as file:
        json.dump(test_data, file, indent=4)

extract_response_and_save(responses_file.parent, responses_file.name)

## 7 · Evaluate & reflect
Start local Qwen on your Mac before running this section. Scoring is skipped on Runpod.

To grade saved responses: put the JSON in the matching Mac run folder and use the local kernel. Run **Goal & config → `format_input` function cell only → this section**. No GPT-2 loading or training is needed.

The score measures response quality (0–100), not classification accuracy.

#### Grade saved responses

In [ ]:
def generate_model_scores(data, judge_model, think=None):
    scores = []
    for entry in tqdm(data, desc="Scoring"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        reply = query_model(prompt, model=judge_model, think=think)
        try:
            entry["judge_score"] = int(reply)
            scores.append(entry["judge_score"])
        except ValueError:
            entry["judge_score"] = None
            print(f"Skipped invalid score: {reply!r}")

    return scores

if not pod_folder.is_dir():
    test_data = json.loads(responses_file.read_text())

    scores = generate_model_scores(test_data, "qwen3.8:27b-mlx", think=False)
    print(f"Scored: {len(scores)}/{len(test_data)}")
    if scores:
        print(f"Average score: {sum(scores) / len(scores):.2f}/100")


    # Save grades
    with open(responses_file, "w") as file:
        json.dump(test_data, file, indent=4)